In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image
import cv2

# 定义输出目录
out_dir = "/home/weiji/restart_exam/20250307/MIROC/attempt"
os.makedirs(out_dir, exist_ok=True)

# 定义生成文件路径的函数
def get_file_paths(data_type, var_name):
    """
    生成 MIROC-ES2H monZ 数据的文件列表：
      - INT: 一整段 200001–208912
      - NINT: 三段 200001–202912, 203001–205912, 206001–208912
    """
    base_path = (
        f"/mnt/backup_ETH/MIROC-ES2H/PD-{data_type}"
        f"/r1i1p1f1/monZ/{var_name}/gn/v20241223/"
    )
    if data_type == 'NINT':
        # NINT 有 3 个成员 / 时间段
        segments = [
            ('r1i1p1f1', '200001', '202912'),
            ('r2i1p1f1', '203001', '205912'),
            ('r3i1p1f1', '206001', '208912'),
        ]
        files = []
        for member, start, end in segments:
            fn = (
                f"{base_path}"
                f"{var_name}_monZ_MIROC-ES2H_PDNINT_{member}_gn_{start}-{end}.nc"
            )
            files.append(fn)
        return files

    else:
        # 比如 data_type='INT'
        fn = (
            f"{base_path}"
            f"{var_name}_monZ_MIROC-ES2H_PD{data_type}_r1i1p1f1_gn_200001-208912.nc"
        )
        return [fn]
        

# 数据处理函数
def process_data(var_name, file_paths, deseasoned=False, lat_range=(-5, 5)):
    """
    处理数据：读取、合并并可选去季节化。
    
    参数：
    - var_name: str, 变量名 ('o3', 'ua', 'ta')
    - file_paths: list, NetCDF文件路径列表
    - deseasoned: bool, 是否去季节化
    - lat_range: tuple, 纬度范围，默认 (-10, 10)
    """
    datasets = [xr.open_dataset(f)[var_name] for f in file_paths]
    data = xr.concat(datasets, dim='time')
    time = data.time
    plev = data.plev
    lat = data.lat
    data_equator = data.sel(lat=slice(lat_range[0], lat_range[1])).mean(dim='lat')
    if deseasoned:
        monthly_clim = data_equator.groupby('time.month').mean('time')
        data_equator = data_equator.groupby('time.month') - monthly_clim
    years = time.dt.year + time.dt.month / 12.0
    plev_hPa = plev / 100.0
    for ds in datasets:
        ds.close()
    return years, plev_hPa, data_equator

def plot_contour(years, plev_hPa, data, title, label, filename, vmin, vmax):
    plt.figure(figsize=(24, 4))
    levels = np.linspace(vmin, vmax, 21)  # 固定levels
    contour = plt.contourf(years, plev_hPa, data.T, levels=levels, cmap='RdBu_r')
    plt.contour(years, plev_hPa, data.T, levels=[0], colors='black', linewidths=1)
    plt.colorbar(contour, label=label)
    plt.xlabel('Year')
    plt.ylabel('Pressure (hPa)')
    plt.yscale('log')
    plt.gca().invert_yaxis()
    plt.gca().set_yticks([1, 5, 10, 20, 50, 100])
    plt.gca().set_ylim(100, 1)
    plt.title(title)
    plt.grid(True, alpha=0.3, which='both')
    plt.savefig(os.path.join(out_dir, filename), dpi=300, bbox_inches='tight')
    plt.show()

# 折线图函数
def plot_line(years_int, data_int, years_nint, data_nint, plev_hPa, plev_target, var_name, title, ylabel, filename):
    """
    生成并保存折线图，绘制INT和NINT的6个月滑动平均。
    
    参数：
    - years_int: array, INT数据的年份
    - data_int: DataArray, INT数据
    - years_nint: array, NINT数据的年份
    - data_nint: DataArray, NINT数据
    - plev_hPa: array, 压力高度 (hPa)
    - plev_target: float, 目标压力层 (hPa)
    - var_name: str, 变量名
    - title: str, 图标题
    - ylabel: str, Y轴标签
    - filename: str, 输出文件名
    """
    # 选择目标压力层的数据
    data_int_level = data_int.sel(plev=plev_target * 100, method='nearest').to_pandas()
    data_nint_level = data_nint.sel(plev=plev_target * 100, method='nearest').to_pandas()
    
    # 计算6个月滑动平均
    data_int_smooth = data_int_level.rolling(window=6, center=True).mean()
    data_nint_smooth = data_nint_level.rolling(window=6, center=True).mean()
    
    plt.figure(figsize=(24, 4))
    plt.plot(years_int, data_int_smooth, label='INT', color='blue')
    plt.plot(years_nint, data_nint_smooth, label='NINT', color='red')
    plt.xlabel('Year')
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(os.path.join(out_dir, filename), dpi=300, bbox_inches='tight')
    plt.show()

# 为INT和NINT数据生成等高线图
for data_type in ['INT', 'NINT']:
    # 非去季节化的平均东风（eastward wind）
    ua_files = get_file_paths(data_type, 'ua')
    ua_years, ua_plev_hPa, ua_data = process_data('ua', ua_files, deseasoned=False)
    plot_contour(
        ua_years, ua_plev_hPa, ua_data,
        f'Equatorial Mean Eastward Wind (100 hPa to 1 hPa) - {data_type}',
        'Eastward Wind (m/s)',
        f'equatorial_eastward_wind_mean_{data_type}.png',
        vmin=-40, vmax=40
    )
    
    # 去季节化的臭氧摩尔分数异常
    o3_files = get_file_paths(data_type, 'o3')
    o3_years, o3_plev_hPa, o3_data = process_data('o3', o3_files, deseasoned=True)
    plot_contour(
        o3_years, o3_plev_hPa, o3_data,
        f'Equatorial Ozone Molar Fraction Anomalies (100 hPa to 1 hPa) - {data_type}',
        'Ozone Anomalies (mol/mol)',
        f'equatorial_ozone_anomalies_{data_type}.png',
        vmin=-1.15e-6, vmax=1.15e-6
    )
    
    # 去季节化的气温异常
    ta_files = get_file_paths(data_type, 'ta')
    ta_years, ta_plev_hPa, ta_data = process_data('ta', ta_files, deseasoned=True)
    plot_contour(
        ta_years, ta_plev_hPa, ta_data,
        f'Equatorial Air Temperature Anomalies (100 hPa to 1 hPa) - {data_type}',
        'Temperature Anomalies (K)',
        f'equatorial_temperature_anomalies_{data_type}.png',
        vmin=-6, vmax=6
    )

# 为折线图准备数据
ua_int_years, ua_int_plev_hPa, ua_int_data = process_data('ua', get_file_paths('INT', 'ua'), deseasoned=False)
ua_nint_years, ua_nint_plev_hPa, ua_nint_data = process_data('ua', get_file_paths('NINT', 'ua'), deseasoned=False)
o3_int_years, o3_int_plev_hPa, o3_int_data = process_data('o3', get_file_paths('INT', 'o3'), deseasoned=False)
o3_nint_years, o3_nint_plev_hPa, o3_nint_data = process_data('o3', get_file_paths('NINT', 'o3'), deseasoned=False)
ta_int_years, ta_int_plev_hPa, ta_int_data = process_data('ta', get_file_paths('INT', 'ta'), deseasoned=False)
ta_nint_years, ta_nint_plev_hPa, ta_nint_data = process_data('ta', get_file_paths('NINT', 'ta'), deseasoned=False)

# 定义目标压力层
plev_targets = [10, 20, 30, 50]

# 绘制折线图
for plev in plev_targets:
    # ua折线图
    plot_line(
        ua_int_years, ua_int_data, ua_nint_years, ua_nint_data,
        ua_int_plev_hPa, plev, 'ua',
        f'Equatorial Mean Eastward Wind at {plev} hPa (6-Month Running Mean)',
        'Eastward Wind (m/s)',
        f'eastward_wind_{plev}hPa_line.png'
    )
    # o3折线图
    plot_line(
        o3_int_years, o3_int_data, o3_nint_years, o3_nint_data,
        o3_int_plev_hPa, plev, 'o3',
        f'Equatorial Ozone Anomalies at {plev} hPa (6-Month Running Mean)',
        'Ozone Anomalies (mol/mol)',
        f'ozone_{plev}hPa_line.png'
    )
    # ta折线图
    plot_line(
        ta_int_years, ta_int_data, ta_nint_years, ta_nint_data,
        ta_int_plev_hPa, plev, 'ta',
        f'Equatorial Temperature Anomalies at {plev} hPa (6-Month Running Mean)',
        'Temperature Anomalies (K)',
        f'temperature_{plev}hPa_line.png'
    )


In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image
import cv2

# 定义输出目录
out_dir = "/home/weiji/restart_exam/20250307/MIROC/attempt"
os.makedirs(out_dir, exist_ok=True)

# 定义生成文件路径的函数
def get_file_paths(data_type, var_name):
    base_path1 = f"/mnt/backup_ETH/MIROC-ES2H/PD-{data_type}/r1i1p1f1/AmonZ/{var_name}/gn/v20240220/"
    base_path2 = f"/mnt/backup_ETH/MIROC-ES2H/PD-{data_type}/r1i1p1f1/AmonZ/{var_name}/gn/v20240517/"
    files = [
        f"{base_path1}{var_name}_AmonZ_MIROC-ES2H_PD-{data_type}_r1i1p1f1_gn_202001-206912.nc",
        f"{base_path2}{var_name}_AmonZ_MIROC-ES2H_PD-{data_type}_r1i1p1f1_gn_207001-207412.nc"
    ]
    return files

# 数据处理函数
def process_data_for_2d(var_name, file_paths, lat_range=(-30, 30), plev_range=(100, 10), 
                        start_time=None, end_time=None):
    """
    处理数据用于二维剖面图，保留纬度维度，并在数据处理阶段完成范围选择。
    
    参数：
    - var_name: str, 变量名 ('ua', 'o3', 'ta')
    - file_paths: list, NetCDF文件路径列表
    - lat_range: tuple, 纬度范围，默认 (-30, 30)
    - plev_range: tuple, 压力范围 (hPa)，默认 (100, 10)
    - start_time: str, 开始时间 (e.g., '2063-01'), 默认None表示数据开头
    - end_time: str, 结束时间 (e.g., '2069-12'), 默认None表示数据结尾
    
    返回：
    - data: DataArray, 处理后的数据，包含time, plev, lat维度
    """
    # 读取并合并数据集
    datasets = [xr.open_dataset(f)[var_name] for f in file_paths]
    data = xr.concat(datasets, dim='time')
    
    # 转换压力范围到Pa
    plev_min, plev_max = plev_range[1] * 100, plev_range[0] * 100  # hPa to Pa
    
    # 选择纬度和压力范围
    data_subset = data.sel(lat=slice(lat_range[0], lat_range[1]), 
                          plev=slice(plev_max, plev_min))
    
    # 处理时间范围
    if start_time and end_time:
        data_subset = data_subset.sel(time=slice(start_time, end_time))
    
    # 关闭数据集释放内存
    for ds in datasets:
        ds.close()
    
    return data_subset



def plot_vertical_profile_images(ua_int_data, ua_nint_data, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    times = ua_int_data.time
    plev_hPa = ua_int_data.plev / 100.0
    lat = ua_int_data.lat
    
    # 定义固定的 levels，确保色标范围固定在 -30 到 30
    levels = np.linspace(-30, 30, 21)
    
    for frame in range(len(times)):
        fig, (ax_int, ax_nint) = plt.subplots(1, 2, figsize=(20, 6), sharey=True)
        
        data_int = ua_int_data.isel(time=frame)
        data_nint = ua_nint_data.isel(time=frame)
        
        # 裁剪数据到 -30 到 30（可选，防止超出范围）
        data_int_clipped = data_int.clip(min=-30, max=30)
        data_nint_clipped = data_nint.clip(min=-30, max=30)
        
        # 绘制等高线图，使用固定的 levels
        contour_int = ax_int.contourf(lat, plev_hPa, data_int_clipped, levels=levels, cmap='RdBu_r', extend='both')
        ax_int.contour(lat, plev_hPa, data_int, levels=[0], colors='black', linewidths=1)
        contour_nint = ax_nint.contourf(lat, plev_hPa, data_nint_clipped, levels=levels, cmap='RdBu_r', extend='both')
        ax_nint.contour(lat, plev_hPa, data_nint, levels=[0], colors='black', linewidths=1)
        
        # 添加 colorbar，不需要 vmin 和 vmax 参数
        cbar_int = plt.colorbar(contour_int, ax=ax_int, label='Mean U Wind (m/s)')
        cbar_nint = plt.colorbar(contour_nint, ax=ax_nint, label='Mean U Wind (m/s)')
        
        # 设置坐标轴
        for ax in (ax_int, ax_nint):
            ax.set_xlabel('Latitude (°N)')
            ax.set_ylabel('Pressure (hPa)')
            ax.set_yscale('log')
            ax.invert_yaxis()
            ax.set_ylim(100, 10)
            ax.set_yticks([10, 100])
        
        # 设置标题
        year = int(data_int.time.dt.year)
        month = int(data_int.time.dt.month)
        ax_int.set_title(f'INT - {year}-{month:02d}')
        ax_nint.set_title(f'NINT - {year}-{month:02d}')
        
        # 添加最大值和最小值文本
        max_val_int = float(data_int.max())
        min_val_int = float(data_int.min())
        max_val_nint = float(data_nint.max())
        min_val_nint = float(data_nint.min())
        ax_int.text(-0.15, 0.95, f'Max: {max_val_int:.2f} m/s', transform=ax_int.transAxes, color='black', fontsize=10)
        ax_int.text(-0.15, 0.90, f'Min: {min_val_int:.2f} m/s', transform=ax_int.transAxes, color='black', fontsize=10)
        ax_nint.text(-0.15, 0.95, f'Max: {max_val_nint:.2f} m/s', transform=ax_nint.transAxes, color='black', fontsize=10)
        ax_nint.text(-0.15, 0.90, f'Min: {min_val_nint:.2f} m/s', transform=ax_nint.transAxes, color='black', fontsize=10)
        
        # 保存图像
        output_filename = os.path.join(output_dir, f'frame_{frame:04d}.png')
        plt.savefig(output_filename)
        plt.close()

# 生成视频的函数
def create_video_from_images(image_dir, output_filename, fps=1.25):
    """
    从图像文件夹中的PNG文件创建MP4视频。
    
    参数：
    - image_dir: str, 图像文件夹路径
    - output_filename: str, 输出视频文件名
    - fps: float, 视频帧率（每秒帧数）
    """
    # 获取所有PNG文件并排序
    images = [img for img in os.listdir(image_dir) if img.endswith(".png")]
    images.sort()
    
    # 读取第一张图像以获取尺寸
    first_image = Image.open(os.path.join(image_dir, images[0]))
    width, height = first_image.size
    
    # 创建视频写入器
    video_path = os.path.join(out_dir, output_filename)
    video = cv2.VideoWriter(video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    
    for image in images:
        img_path = os.path.join(image_dir, image)
        frame = cv2.imread(img_path)
        video.write(frame)
    
    video.release()

# 为INT和NINT数据准备二维剖面数据并生成视频
ua_int_files = get_file_paths('INT', 'ua')
ua_nint_files = get_file_paths('NINT', 'ua')

# 第一组视频：2063年1月到2069年12月
ua_int_data_2d_1 = process_data_for_2d('ua', ua_int_files, lat_range=(-30, 30), plev_range=(100, 10), 
                                       start_time='2063-01', end_time='2069-12')
ua_nint_data_2d_1 = process_data_for_2d('ua', ua_nint_files, lat_range=(-30, 30), plev_range=(100, 10), 
                                        start_time='2063-01', end_time='2069-12')
image_dir_1 = os.path.join(out_dir, 'images_2063-2069')
plot_vertical_profile_images(ua_int_data_2d_1, ua_nint_data_2d_1, image_dir_1)
create_video_from_images(image_dir_1, 'vertical_profile_2063-2069_INT_NINT.mp4')

# 第二组视频：2036年8月到2040年10月
ua_int_data_2d_2 = process_data_for_2d('ua', ua_int_files, lat_range=(-30, 30), plev_range=(100, 10), 
                                       start_time='2036-08', end_time='2040-10')
ua_nint_data_2d_2 = process_data_for_2d('ua', ua_nint_files, lat_range=(-30, 30), plev_range=(100, 10), 
                                        start_time='2036-08', end_time='2040-10')
image_dir_2 = os.path.join(out_dir, 'images_2036-2040')
plot_vertical_profile_images(ua_int_data_2d_2, ua_nint_data_2d_2, image_dir_2)
create_video_from_images(image_dir_2, 'vertical_profile_2036-2040_INT_NINT.mp4')

# 第三组视频：2045年12月到2049年11月
ua_int_data_2d_3 = process_data_for_2d('ua', ua_int_files, lat_range=(-30, 30), plev_range=(100, 10), 
                                       start_time='2045-12', end_time='2049-11')
ua_nint_data_2d_3 = process_data_for_2d('ua', ua_nint_files, lat_range=(-30, 30), plev_range=(100, 10), 
                                        start_time='2045-12', end_time='2049-11')
image_dir_3 = os.path.join(out_dir, 'images_2045-2049')
plot_vertical_profile_images(ua_int_data_2d_3, ua_nint_data_2d_3, image_dir_3)
create_video_from_images(image_dir_3, 'vertical_profile_2045-2049_INT_NINT.mp4')

In [ ]:
import os
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
from scipy.signal import welch

# 设置输出目录
out_dir = "/home/weiji/restart_exam/20250307/MIROC/attempt2"
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

# 数据处理函数
def process_data(var_name, file_paths, lat_range=(-10, 10)):
    """
    处理数据：读取、合并。
    
    参数：
    - var_name: str, 变量名 ('o3', 'ua', 'ta')
    - file_paths: list, NetCDF文件路径列表
    - lat_range: tuple, 纬度范围，默认 (-10, 10)
    """
    datasets = [xr.open_dataset(f)[var_name] for f in file_paths]
    data = xr.concat(datasets, dim='time')
    time = data.time
    plev = data.plev
    lat = data.lat
    data_equator = data.sel(lat=slice(lat_range[0], lat_range[1])).mean(dim='lat')
    years = time.dt.year + time.dt.month / 12.0
    plev_hPa = plev / 100.0
    for ds in datasets:
        ds.close()
    return years, plev_hPa, data_equator

# 假设 get_file_paths 已经定义
ua_int_years, ua_int_plev_hPa, ua_int_data = process_data('ua', get_file_paths('INT', 'ua'))
ua_nint_years, ua_nint_plev_hPa, ua_nint_data = process_data('ua', get_file_paths('NINT', 'ua'))

# 折线图函数
def plot_line(years_int, data_int, years_nint, data_nint, plev_hPa, plev_target, var_name, out_dir):
    """
    生成并保存折线图，绘制INT和NINT的原始数据。
    
    参数：
    - years_int: array, INT数据的年份
    - data_int: DataArray, INT数据
    - years_nint: array, NINT数据的年份
    - data_nint: DataArray, NINT数据
    - plev_hPa: array, 压力高度 (hPa)
    - plev_target: float, 目标压力层 (hPa)
    - var_name: str, 变量名
    - out_dir: str, 输出目录
    """
    data_int_level = data_int.sel(plev=plev_target * 100, method='nearest').to_pandas()
    data_nint_level = data_nint.sel(plev=plev_target * 100, method='nearest').to_pandas()
    
    plt.figure(figsize=(24, 4))
    plt.plot(years_int, data_int_level, label='INT', color='blue')
    plt.plot(years_nint, data_nint_level, label='NINT', color='red')
    plt.xlabel('Year')
    plt.ylabel('Wind Speed (m/s)')
    plt.title(f'QBO: Zonal Wind (ua) at {plev_target} hPa')
    plt.legend()
    plt.grid(True, alpha=0.3)
    filename = f'ua_line_{plev_target}hPa.png'
    plt.savefig(os.path.join(out_dir, filename), dpi=300, bbox_inches='tight')
    plt.show()  # 关闭图形以释放内存

# 修改后的标注函数，增加了左右偏移和颜色参数，同时将y方向偏移设为负值使其往下显示
def annotate_peaks(xf, yf, ax, top_n=1, offset_x=0, color='black'):
    """
    标注功率谱中的前 top_n 个波峰，并根据参数设置文字颜色及偏移量。
    
    参数：
    - xf: array, 频率
    - yf: array, 功率谱密度
    - ax: matplotlib.axes, 坐标轴对象
    - top_n: int, 标注前几个波峰
    - offset_x: float, x方向偏移（单位：点）
    - color: str, 标注颜色
    """
    peak_indices = np.argsort(yf)[-top_n:][::-1]  # 获取前 top_n 个波峰的索引
    for idx in peak_indices:
        freq = xf[idx]
        psd_value = yf[idx]
        if freq > 0:  # 只标注非零频率
            period = 1 / freq
            ax.annotate(f'{period:.1f} mo', (freq, psd_value),
                        textcoords="offset points", xytext=(offset_x, -10),
                        ha='center', fontsize=9, color=color)
            ax.scatter(freq, psd_value, color=color, s=50, zorder=5)

# 功率谱绘制函数，调整了标签、标题以及调用标注函数时传入相应偏移量和颜色
def plot_psd(years_list, data_list, plev_hPa, plev_target, var_name, case_list, out_dir):
    """
    使用Welch方法计算并绘制INT和NINT的功率谱。
    
    参数：
    - years_list: list of arrays, 年份数据列表 [years_int, years_nint]
    - data_list: list of DataArrays, 数据列表 [data_int, data_nint]
    - plev_hPa: array, 压力高度 (hPa)
    - plev_target: float, 目标压力层 (hPa)
    - var_name: str, 变量名
    - case_list: list of str, 数据类型列表 ['INT', 'NINT']
    - out_dir: str, 输出目录
    """
    fig, ax = plt.subplots(figsize=(5, 6))

    for years, data, case in zip(years_list, data_list, case_list):
        data_level = data.sel(plev=plev_target * 100, method='nearest').to_numpy()
        dt = 1  # 采样间隔为1个月

        # 使用Welch方法计算功率谱密度
        freqs, psd = welch(data_level, fs=1/dt, window='hann', nperseg=660)
        
        # 绘制功率谱，并捕获绘图返回的line对象以获取颜色
        line, = ax.plot(freqs, psd, label=f'{case} Power Spectrum')
        line_color = line.get_color()
        
        # 根据case设置标注偏移，INT向左偏移，NINT向右偏移
        if case.upper() == 'INT':
            offset_x = -30
        elif case.upper() == 'NINT':
            offset_x = 30
        else:
            offset_x = 0
        
        # 标注波峰，并传入对应颜色和偏移量
        annotate_peaks(freqs, psd, ax, top_n=1, offset_x=offset_x, color=line_color)

    # 调整X、Y轴标签及标题，突出QBO研究背景
    ax.set_xlabel('Frequency (cycles/month)')
    ax.set_ylabel('Power Spectral Density ((m/s)²)')
    ax.set_title(f'QBO: Zonal Wind (ua) Power Spectrum at {plev_target} hPa')
    ax.set_xlim(0, 0.08)  
    ax.legend()
    ax.grid(True, alpha=0.3)
    filename_psd = f'{var_name}_power_spectrum_{plev_target}hPa.png'
    fig.savefig(os.path.join(out_dir, filename_psd), dpi=300)
    plt.show(fig)  # 关闭图形以释放内存

# 压力层列表
plev_targets = [10, 20, 30, 50]

# 主循环
for plev_target in plev_targets:
    # 绘制折线图
    plot_line(ua_int_years, ua_int_data, ua_nint_years, ua_nint_data, ua_int_plev_hPa, plev_target, 'ua', out_dir)
    # 绘制功率谱
    plot_psd(
        years_list=[ua_int_years, ua_nint_years],
        data_list=[ua_int_data, ua_nint_data],
        plev_hPa=ua_int_plev_hPa,
        plev_target=plev_target,
        var_name='ua',
        case_list=['INT', 'NINT'],
        out_dir=out_dir
    )


In [ ]:
# 新增 block：温度 (ta) 和臭氧 (o3) 的傅里叶变换（原始数据 vs 去趋势数据）
# 使用目标压力层：10, 20, 30, 50 hPa
def process_data(var_name, file_paths, deseasoned=False, lat_range=(-5, 5)):
    """
    处理数据：读取、合并并可选去季节化。
    
    参数：
    - var_name: str, 变量名 ('o3', 'ua', 'ta')
    - file_paths: list, NetCDF文件路径列表
    - deseasoned: bool, 是否去季节化
    - lat_range: tuple, 纬度范围，默认 (-10, 10)
    """
    datasets = [xr.open_dataset(f)[var_name] for f in file_paths]
    data = xr.concat(datasets, dim='time')
    time = data.time
    plev = data.plev
    lat = data.lat
    data_equator = data.sel(lat=slice(lat_range[0], lat_range[1])).mean(dim='lat')
    if deseasoned:
        monthly_clim = data_equator.groupby('time.month').mean('time')
        data_equator = data_equator.groupby('time.month') - monthly_clim
    years = time.dt.year + time.dt.month / 12.0
    plev_hPa = plev / 100.0
    for ds in datasets:
        ds.close()
    return years, plev_hPa, data_equator

# 新增 block：对比 INT 和 NINT 的温度 (ta) 与臭氧 (o3) 傅里叶变换（均使用去季节趋势数据）
# 新增绘图函数：plot_psd_new
def plot_psd_new(years_list, data_list, plev_hPa, plev_target, var_name, case_list, out_dir):
    """
    使用Welch方法计算并绘制去季节趋势后的 INT 和 NINT 数据的功率谱图，
    针对温度 (ta) 或臭氧 (o3)。
    
    参数：
    - years_list: list of arrays, 年份数据列表 [years_INT, years_NINT]
    - data_list: list of DataArrays, 数据列表 [data_INT, data_NINT]
    - plev_hPa: array, 压力高度 (hPa)
    - plev_target: float, 目标压力层 (hPa)
    - var_name: str, 变量名 ('ta' 或 'o3')
    - case_list: list of str, 数据类型列表，例如 ['INT', 'NINT']
    - out_dir: str, 输出目录
    """
    fig, ax = plt.subplots(figsize=(5, 6))
    for years, data, case in zip(years_list, data_list, case_list):
        # 选择目标压力层的数据（注意：数据中的 plev 单位为 Pa，因此转换为 hPa）
        data_level = data.sel(plev=plev_target * 100, method='nearest').to_numpy()
        dt = 1  # 采样间隔为1个月
        # 使用Welch方法计算功率谱密度
        freqs, psd = welch(data_level, fs=1/dt, window='hann', nperseg=660)
        line, = ax.plot(freqs, psd, label=f'{case} Power Spectrum')
        line_color = line.get_color()
        # 设置标注偏移：INT 向左，NINT 向右
        if case.upper() == 'INT':
            offset_x = -30
        elif case.upper() == 'NINT':
            offset_x = 30
        else:
            offset_x = 0
        annotate_peaks(freqs, psd, ax, top_n=1, offset_x=offset_x, color=line_color)
    
    ax.set_xlabel('Frequency (cycles/month)')
    # 根据变量修改y轴标签和图标题
    if var_name == 'ta':
        ylabel = 'Power Spectral Density ((K)²)'
        title_text = f'QBO: Air Temperature Anomalies Power Spectrum at {plev_target} hPa'
    elif var_name == 'o3':
        ylabel = 'Power Spectral Density ((mol/mol)²)'
        title_text = f'QBO: Ozone Molar Fraction Anomalies Power Spectrum at {plev_target} hPa'
    else:
        ylabel = 'Power Spectral Density'
        title_text = f'QBO: {var_name} Power Spectrum at {plev_target} hPa'
    
    ax.set_ylabel(ylabel)
    ax.set_title(title_text)
    ax.set_xlim(0, 0.5)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 根据变量名构造输出文件名
    filename_psd = f'{var_name}_power_spectrum_{plev_target}hPa.png'
    fig.savefig(os.path.join(out_dir, filename_psd), dpi=300)
    plt.show(fig)

# 使用新绘图函数对比 INT 与 NINT 的温度 (ta) 和臭氧 (o3) 数据的傅里叶变换（均采用去季节趋势数据）
# 定义新的输出目录，并创建它
out_dir_fourier2 = "/home/weiji/restart_exam/20250307/MIROC/attempt3"
os.makedirs(out_dir_fourier2, exist_ok=True)

# 定义目标压力层列表
plev_targets = [10, 20, 30, 50]

for var in ['ta', 'o3']:
    # 处理 INT 数据（去季节趋势）
    file_paths_int = get_file_paths('INT', var)
    years_int, plev_hPa, data_int = process_data(var, file_paths_int, deseasoned=False)
    
    # 处理 NINT 数据（去季节趋势）
    file_paths_nint = get_file_paths('NINT', var)
    years_nint, _, data_nint = process_data(var, file_paths_nint, deseasoned=False)
    
    # 针对每个目标压力层绘制傅里叶变换功率谱对比图
    for plev_target in plev_targets:
        plot_psd_new(
            years_list=[years_int, years_nint],
            data_list=[data_int, data_nint],
            plev_hPa=plev_hPa,
            plev_target=plev_target,
            var_name=var,
            case_list=['INT', 'NINT'],
            out_dir=out_dir_fourier2
        )




In [ ]:
# 新的 Block：基于30 hPa处U值筛选时间点（提取满足条件的时间坐标），计算各变量的垂直平均并绘制图形
# 注意：本 block 依赖之前已运行的代码中的变量（如 ua_int_data, ua_nint_data 等）
# 此外，温度图需要使用去季节化（anomaly）数据，因此重新处理温度数据时需设定 deseasoned=True

# 定义新的输出目录
new_out_dir = "/home/weiji/restart_exam/20250307/MIROC/attempt4"
os.makedirs(new_out_dir, exist_ok=True)

ua_int_30 = ua_int_data.sel(plev=5000, method='nearest')
ua_nint_30 = ua_nint_data.sel(plev=5000, method='nearest')


# -------------------------------
# Step 1: 提取30 hPa处（3000 Pa）U值满足条件的时间坐标
# 对于 INT 数据（U数据使用原始数据即可）：
times_qbow_int = ua_int_30.time.values[ua_int_30.values > 5]    # U > 0 的时刻（QBO W）
times_qboe_int = ua_int_30.time.values[ua_int_30.values < -2.5]   # U < -10 的时刻（QBO E）
# 对于 NINT 数据：
times_qbow_nint = ua_nint_30.time.values[ua_nint_30.values > 5]    # U > 0 的时刻（QBO W）
times_qboe_nint = ua_nint_30.time.values[ua_nint_30.values < -2.5]   # U < -10 的时刻（QBO E）

# -------------------------------
# Step 2: 利用上述时间坐标从完整数据中筛选相应时间点的数据（风速）
qbow_int = ua_int_data.sel(time=times_qbow_int)
qbow_nint = ua_nint_data.sel(time=times_qbow_nint)
qboe_int = ua_int_data.sel(time=times_qboe_int)
qboe_nint = ua_nint_data.sel(time=times_qboe_nint)

# -------------------------------
# Step 3: 计算筛选后风速数据的垂直平均（沿时间维度求平均，每个压力层得到一个平均值）
mean_ua_int_qbow = qbow_int.mean(dim='time')
mean_ua_nint_qbow = qbow_nint.mean(dim='time')
mean_ua_int_qboe = qboe_int.mean(dim='time')
mean_ua_nint_qboe = qboe_nint.mean(dim='time')

mean_difference_int = mean_ua_int_qbow - mean_ua_int_qboe
mean_difference_nint = mean_ua_nint_qbow - mean_ua_nint_qboe


# -------------------------------
# Step 4: 定义绘图函数（绘制INT和NINT曲线，Y轴为气压，X轴为变量值，且X=0垂直线位于中间）
def plot_profile(mean_int, mean_nint, x_label, title_text, filename, xlim_fixed=None):
    """
    绘制包含INT和NINT两条曲线的垂直平均图：
    - Y轴为气压（单位 hPa，范围 1-100 hPa，使用对数刻度并反转）
    - X轴为变量值（风速或温度），并在X=0处绘制垂直虚线
    """
    plt.figure(figsize=(6, 8))
    # 转换压力单位：Pa -> hPa
    plev_hPa = mean_int.plev / 100.0

    # 绘制 INT 和 NINT 曲线（使用 marker 便于观察）
    plt.plot(mean_int.values, plev_hPa, label='INT', marker='o')
    plt.plot(mean_nint.values, plev_hPa, label='NINT', marker='o')

    # 添加 X=0 处的垂直虚线
    plt.axvline(0, color='black', linewidth=1, linestyle='--')
    plt.xlabel(x_label)
    plt.ylabel('Pressure (hPa)')
    plt.title(title_text)
    plt.yscale('log')
    plt.gca().invert_yaxis()
    plt.ylim(100, 1)

    # 设置 X 轴为对称范围，确保 X=0 位于图像正中
    if xlim_fixed is not None:
        plt.xlim(xlim_fixed)
    else:
        max_val = max(abs(mean_int.values).max(), abs(mean_nint.values).max())
        plt.xlim([-max_val * 1.1, max_val * 1.1])
    
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(os.path.join(new_out_dir, filename), dpi=300, bbox_inches='tight')
    plt.show()

# -------------------------------
# Step 5: 绘制风速的垂直平均图

# QBO W (U > 0)——绘制 INT 和 NINT 的 U 平均曲线（固定 X 轴范围为 [-40, 40]）
plot_profile(
    mean_int=mean_ua_int_qbow,
    mean_nint=mean_ua_nint_qbow,
    x_label='U (m/s)',
    title_text='QBO W (30 hPa: U > 0)',
    filename='qbow_u_profile.png',
    xlim_fixed=(-40, 40)
)

# QBO E (U < -10)——绘制 INT 和 NINT 的 U 平均曲线（固定 X 轴范围为 [-40, 40]）
plot_profile(
    mean_int=mean_ua_int_qboe,
    mean_nint=mean_ua_nint_qboe,
    x_label='U (m/s)',
    title_text='QBO E (30 hPa: U < -10)',
    filename='qboe_u_profile.png',
    xlim_fixed=(-40, 40)
)

plot_profile(
    mean_int=mean_difference_int,
    mean_nint=mean_difference_nint,
    x_label='U (m/s)',
    title_text='QBO E (30 hPa: U < -10)',
    filename='qboe_u_profile.png',
    xlim_fixed=(-40, 40)
)

# -------------------------------
# Step 6: 温度异常数据处理与绘图（温度去季节化 anomaly）
# 重新处理温度数据，利用 deseasoned=True 计算温度异常数据
_, _, ta_int_anom = process_data('ta', get_file_paths('INT', 'ta'), deseasoned=True)
_, _, ta_nint_anom = process_data('ta', get_file_paths('NINT', 'ta'), deseasoned=True)

# 利用相同的时间坐标（从 U 数据提取）筛选温度异常数据
qbow_int_ta = ta_int_anom.sel(time=times_qbow_int)
qbow_nint_ta = ta_nint_anom.sel(time=times_qbow_nint)
qboe_int_ta = ta_int_anom.sel(time=times_qboe_int)
qboe_nint_ta = ta_nint_anom.sel(time=times_qboe_nint)

# 计算温度异常数据的垂直平均
mean_ta_int_qbow = qbow_int_ta.mean(dim='time')
mean_ta_nint_qbow = qbow_nint_ta.mean(dim='time')
mean_ta_int_qboe = qboe_int_ta.mean(dim='time')
mean_ta_nint_qboe = qboe_nint_ta.mean(dim='time')

# 绘制温度异常垂直平均图
plot_profile(
    mean_int=mean_ta_int_qbow,
    mean_nint=mean_ta_nint_qbow,
    x_label='Temperature Anomaly (K)',
    title_text='QBO W Temperature Anomaly Average',
    filename='qbow_ta_anom_profile.png'
)

plot_profile(
    mean_int=mean_ta_int_qboe,
    mean_nint=mean_ta_nint_qboe,
    x_label='Temperature Anomaly (K)',
    title_text='QBO E Temperature Anomaly Average',
    filename='qboe_ta_anom_profile.png'
)
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os

# 定义输出目录
out_dir = "/home/weiji/restart_exam/20250307/MIROC/attempt5"
os.makedirs(out_dir, exist_ok=True)

# 假设 get_file_paths 和 process_data 已经定义好，用于读取数据
# 获取 INT 数据的风速（ua）
ua_files = get_file_paths('INT', 'ua')
ua_years, ua_plev_hPa, ua_data = process_data('ua', ua_files, deseasoned=False)

# 注意：ua_data 的 plev 单位为 Pa，因此 30 hPa 对应 3000 Pa
ua_30 = ua_data.sel(plev=3000, method='nearest')

# 构建 QBO E mask：在 30 hPa 层满足 U < -10 m/s 的时刻标记为 True
qboe_mask = ua_30 < -10

# 对 ua_data 所有压力层，在非 QBO E 时刻填入 NaN（保留原有时间和压力坐标）
ua_data_qboe = ua_data.where(qboe_mask, other=np.nan)

# 绘制垂直剖面等高线图：X轴为年份，Y轴为气压
plt.figure(figsize=(48, 3))
levels = np.linspace(-40, 40, 21)  # 可根据实际数据调整等高线级数
contour = plt.contourf(ua_years, ua_plev_hPa, ua_data_qboe.T, levels=levels, cmap='RdBu_r')
plt.contour(ua_years, ua_plev_hPa, ua_data_qboe.T, levels=[0], colors='black', linewidths=1)
plt.colorbar(contour, label='Eastward Wind (m/s)')
plt.xlabel('Year')
plt.ylabel('Pressure (hPa)')
plt.yscale('log')
plt.gca().invert_yaxis()  # 使高气压在下，低气压在上
plt.gca().set_yticks([1, 5, 10, 20, 50, 100])
plt.ylim(100, 1)  # 限制在 1-100 hPa 范围内
plt.axhline(50, color='red', linewidth=2, linestyle='-')  # 用红色直线标记50 hPa
plt.title('QBO E: INT Equatorial Mean Eastward Wind\n(U < -10 m/s at 30 hPa; non-QBO E times are NaN)')
plt.grid(True, alpha=0.3, which='both')
plt.savefig(os.path.join(out_dir, 'qboe_wind_vertical_profile_INT.png'), dpi=300, bbox_inches='tight')
plt.show()


# 假设 get_file_paths 和 process_data 已经定义好，用于读取数据
# 获取 INT 数据的风速（ua）
ua_files = get_file_paths('NINT', 'ua')
ua_years, ua_plev_hPa, ua_data = process_data('ua', ua_files, deseasoned=False)

# 注意：ua_data 的 plev 单位为 Pa，因此 30 hPa 对应 3000 Pa
ua_30 = ua_data.sel(plev=3000, method='nearest')

# 构建 QBO E mask：在 30 hPa 层满足 U < -10 m/s 的时刻标记为 True
qboe_mask = ua_30 < -10

# 对 ua_data 所有压力层，在非 QBO E 时刻填入 NaN（保留原有时间和压力坐标）
ua_data_qboe = ua_data.where(qboe_mask, other=np.nan)

# 绘制垂直剖面等高线图：X轴为年份，Y轴为气压
plt.figure(figsize=(48, 3))
levels = np.linspace(-40, 40, 21)  # 可根据实际数据调整等高线级数
contour = plt.contourf(ua_years, ua_plev_hPa, ua_data_qboe.T, levels=levels, cmap='RdBu_r')
plt.contour(ua_years, ua_plev_hPa, ua_data_qboe.T, levels=[0], colors='black', linewidths=1)
plt.colorbar(contour, label='Eastward Wind (m/s)')
plt.xlabel('Year')
plt.ylabel('Pressure (hPa)')
plt.yscale('log')
plt.gca().invert_yaxis()  # 使高气压在下，低气压在上
plt.gca().set_yticks([1, 5, 10, 20, 50, 100])
plt.ylim(100, 1)  # 限制在 1-100 hPa 范围内
plt.axhline(50, color='red', linewidth=2, linestyle='-')  # 用红色直线标记50 hPa
plt.title('QBO E: INT Equatorial Mean Eastward Wind\n(U < -10 m/s at 30 hPa; non-QBO E times are NaN)')
plt.grid(True, alpha=0.3, which='both')
plt.savefig(os.path.join(out_dir, 'qboe_wind_vertical_profile_INT.png'), dpi=300, bbox_inches='tight')
plt.show()

